In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# An introduction to inference
We can collect, "analyze", and "visualize" all of the data we want, but a real goal of data science is to _draw conclusions from data_.
Drawing conclusions from data comes in many forms, but one of the most basic is with **statistical inference**.
This process relies on statistical tools to conclude properties of an underlying distribution.
As such, we are trying to advance beyond _descriptive statistics_ (e.g., calculating mean and variance) and towards using data to inform decisions.

This is not meant to be a comprehensive introduction to statistical inference.
There are textbook (chapter) and significant portions of classes dedicated to the nuances of the topic.
We are going to work towards a minimal working understanding so that we can start to use Python for inferential statistics and interpret its outputs.

We will continue with our geometric dice-rolling example.
In this scenario, you encounter a stranger on the street who invites you to play his dice rolling game.
Your spidey-senses tingle and start to wonder if the dice he gives you is fair...

## Key aspects to inference
Much like "doing good science" adheres to the scientific method, statistical inference admits a proper protocol.
In short, proper steps in statistical inference include:
* **Formulate a question** Is the dice handed to you fair?
* **Determine current knowledge base** In this case, we know the dice-rolling game follows a geometric distribution.
* **Form a testable hypothesis** This requires nuance in statistical inference, but you want to come up with a statement that is testable. For example, the coin is fair. Or, the coin is not fair.
* **Collect Data** You might observe other people play the game.
* **Analyze the data** This could include descriptive statistics or visualizations. For example, what is the distribution of all the games you observe?
* **Make a conclusion** Statistical inference falls here (and partially in the above step). But we want to come to a clearly stated conclusion based on the analysis of our data.

The above order is important; in particular, you want to form a hypothesis before you collect the data.
If you reverse the order, your data collection might _bias_ your hypothesis.

## Key words around statistical inference

**Null Hypothesis** The statement about our study that maintains the expected status quo. For our case, the status quo would be that the dice is fair. We would express this null hypothesis as:
\begin{equation*}
    H_0\colon \mathbf{P} = \frac{1}{6},
\end{equation*}
where $P$ is the probability of success.
In most cases, the Null hypothesis contains an equality.
This could be "$=$", "$\leq$", or "$\geq$".
In our designs, we set-up the null hypothesis as a statement that we "reject in favor of the alternative hypothesis" or "fail to reject."
If we have collected enough evidence to reject the null hypothesis as a statement, then we have collected enough evidence to go against the "status quo".

**Alternative Hypothesis** This is the statement that opposes the null hypothesis. In the dice example, we would write
\begin{equation*}
    H_a\colon \mathbf{P} \neq \frac{1}{6}.
\end{equation*}
This would confirm our spidey-senses that the dice is not fair.

**Statistically significant** An outcome is considered statistically significant if it is unlikely to occur due to randomness alone, provided the null hypothesis is true.

So, what's "unlikely"? This where we see that null hypothesis as a major driver of our analysis.
The null hypothesis will induce a null distribution. Under this distribution, we will have a notion of what qualifies as an "unlikely" event.

**test statistic** The statistic we compute from data to measure how far the data is from the null hypothesis. In our case, the test statistic is the average of the number of rolls until obtaining a six.

**p-value** A $p$-value is the probability (or proportion) of observing an outcome _as extreme or more extreme_ than our computed statistic under the null hypothesis.

If a $p$-value is low, then not very many outcomes are more extreme than our statistic.
We interpret this to mean that our observed statistic is not very likely to happen, due to chance, when the null is true.
So, we conclude that the null is false, and another description would more accurately characterize our observed data.

It is important that you establish a "level" of significance _before_ computing a $p$-value.
This is often reported as an $\alpha$ level of significance.
For example, we would specify an $\alpha$-level of $0.05$. If we compute a $p$-value larger than 0.05, we fail to reject the null.
If, however, we compute a $p$-value _smaller_ than 0.05, then we reject the null in favor of the alternative and obtain a "statistically significant" result.

## A Central Limit theorem example 
We will contextualize all of this by simulating the dice rolling example. Formally, we will state our hypothesis:
\begin{align*}
    H_0\colon & \ \ \mathbf{P} = \frac{1}{6} \\
    H_a\colon & \ \ \mathbf{P} \neq \frac{1}{6}
\end{align*}
In our simulation, we set represent the null hypothesis with `prob_success_null=1/6` and use this probability to simulate many trials of the dice rolling game.
We compute the mean over each trial, to have a long list of means which we use to build out an _empirical_ estimate to the null distribution.

We set a second probability of success variable, `prob_success_me = ...`, which is the probability of rolling a  6 with the dice the stranger handed to me.
We can set this value to (nearly) whatever we like and compare simulated outcomes with the null distribution.


We set our significance level at $\alpha = 0.05$ and compute the $\alpha/2$ and $1-\alpha/2$ quantiles from the empirical distribution.
That means the data points that lie beyond these values would be considered extreme at the $\alpha$-level.
Due to potential asymmetries in the simulated empirical distribution, these quantiles may not be equi-distant from the empirical or theoretical mean.

We simulate our single trial and compare that with the large simulations.
We count how many of the simulated trials as _as extreme or more extreme_ with a boolean comparison on distance from the mean under the null hypothesis. We consider _more extreme_ results to be trial means that are further away from the mean than our single simulated mean.



In [ ]:
# Set null probability of success to generate a null distribution
prob_success_null = 1/6

# set my probability for testing purposes
prob_success_me = 1/6

# pre-specified alpha level
α = 0.05

# Generate many geometric trials for emperical null distribution
num_sims   = 100
num_trials = 10000
trials = np.random.geometric(prob_success_null, (num_trials, num_sims))

# Calculate mean of each trial
trial_means = np.mean(trials, axis=1)


# Find α/2 and 1-α/2 quantiles in empirical distribution
alpha_quants = np.quantile(trial_means, [α/2, 1-α/2])

# Do my trial
my_trial = np.random.geometric(prob_success_me, num_sims)
my_trial_mean = np.mean( my_trial)


# Find proportion of trial menas as extreme or more extreme than my trial under the null
mean_geo     = 1/prob_success_null
sigma_sq_geo = (1-prob_success_null) / (prob_success_null**2)

mean_CLT     = mean_geo
sigma_sq_CLT = sigma_sq_geo / num_trials

prop = np.sum( np.abs(trial_means - mean_CLT) >= np.abs(my_trial_mean - mean_CLT)) / num_trials

# Make histogram
trial_heights, trial_bins = np.histogram(trial_means, bins=20)

plt.hist(trial_bins[:-1], trial_bins, weights = trial_heights, density=True, label='Trials distribution')

# add lines for quantiles
plt.vlines(x=alpha_quants,ymin=plt.ylim()[0],ymax=plt.ylim()[1], color='cyan',linewidth=2, label='alpha cut-off values')

# add lines for my mean
my_mean_height = np.diff( plt.ylim() ) * 0.1 * [0, 1] + plt.ylim()[0]
plt.vlines(x=my_trial_mean,ymin=my_mean_height[0],ymax=my_mean_height[1], color='orange',linewidth=3, label='my mean')
plt.vlines(x=mean_CLT-(my_trial_mean-mean_CLT),ymin=my_mean_height[0],ymax=my_mean_height[1], color='orange',linewidth=3, 
           linestyle='--',label='my mean reflected')
plt.legend();

plt.xlabel('Mean of %d rolls' %num_sims);
plt.ylabel('Bin frequency (normalized as a density)')

print('My simulated mean:        %0.4f' %my_trial_mean)
print('Alpha cut-off quantiles: [%0.4f, %0.4f]' %(alpha_quants[0],alpha_quants[1]))
print('Proportion of as or more extreme events: %0.4f' %prop)


The above plot shows the empircal distribution and marks all of the computed quantities.
The code also prints the simulated mean, quantiles, and proportion of more extreme events.
Since we have not set a seed for the random number generator, if you run the simulation again, you might get different results, and maybe even arrive at a different reject/fail-to-reject decision!

It is important to remember, that randomness _can_ account for an extreme event.
If you run the above code with `prob_success_me=1/6` enough times, you will have some instances of an incorrect rejection of a null hypothesis.
Hypothesis testing hinges on the basis that the extreme events are _unlikely_ to be explained by chance under the null hypothesis. 

This leads us to two types of errors: 
* Type I error: rejecting a true null hypothesis
* Type II error: failing to reject a false null hypothesis.

The way we set up statistical inference, we accept an $\alpha$-level of Type I errors.
That is, we would expect to make a Type I error at a rate of $\alpha$ under repeated experiments.
So, when we specify an $\alpha$-level of significance, we are specifying an acceptable rate of Type I errors.

## Inference with 1-sample t-tests
We do not have to rely on generating the empirical distribution to perform inference.
We can rely on the theoretical distributions induced by the null hypothesis to produce a single $p$-value that won't change as the simulated empirical distribution changes.

The two first tests students learn are the $z$-test and Student's $t$-test. They are related and both assume an underlying bell-shape to the data.
The $z$-test uses a normal distribution and its pdf, typically transformed to be centered with mean zero.
The $t$-test uses the pdf for Student's $t$-distribution [defined via the Gamma Function](https://en.wikipedia.org/wiki/Student%27s_t-distribution).

The $z$-test is used when the population variance is _known_. In practice, this quantity is rarely known.
The $t$-test is used when the population variance is _unknown_ and the sample data is instead used to obtain a proxy for the population variance.

We will focus with the $t$-test as it is more applicable to most scenarios.
The $t$-test makes the following assumptions:
* The data is approximately normal
* The data follows a symmetric distribution

Since we are working with _sample means_ we meet the normality and symmetric assumptions by the central limit theorem.
It's very important that in this example we are doing inferenc on the _mean_ over a sequenc of dice-rolling-games.
The dice-roll-until-a-six process follows a very different (geometric) distribution; but the mean of a number of samples of the games is expected to be approximately normal.

We can implement the $t$-test simply in Python using the `stats` library from the `scipy` module.
See the code below that uses the `ttest_1samp` function applied to our dice-rolling trial.

In [ ]:
from scipy import stats

t_test_result = stats.ttest_1samp(my_trial, mean_CLT)
print('From 1-sample t-test:')
print('\ttest statistic t* = %0.4f' %t_test_result.statistic)
print('\tp = %0.4f' %t_test_result.pvalue)
print('\tCI = [%0.4f, %0.4f]' %(t_test_result.confidence_interval()[0], t_test_result.confidence_interval()[1]))


Observe the $p$-value is similar to the one obtained from the empirical distribution.
They are not the same; the empirical distribution is only an approximation the the theoretical distribution under the null hypothesis (which converges to the true distribution as the number of simulations gets large).


You will also notice a reported $t$-statistic. This can be thought of as a transformed version of our sample mean.
If our sample mean is below the expected mean under the null, then the $t$-statistic is negative. The $t$-statistic is instead positive if our sample mean is larger than the expected mean.

## Inference with 2-sample t-tests
We might be interested in comparing two datasets.
You will work more with these in an assignment, but the following code will read in Home Run data from a few MLB seasons.
The homerun-per-game-rate is calculated for each team, and we compare the season averages in this quantity between years.


Since there are two samples, there are two means $\mu_1$ and $\mu_2$.
Our null hypothesis is that there is no difference in the averages between the two years,
\begin{align*}
    H_0\colon& \ \ \mu_1-\mu_2 = 0 \\
    H_a\colon& \ \ \mu_1-\mu_2 \neq 0
\end{align*}

Since there are two populations, there are also two (unknown) variances. We can specify `equal_var=True` if we believe the variance to be equal; otherwise we specify `equal_var=False`.

The 2-sample test also comes with assumptions:
* The two samples are independent
* The sample are approximately normal (which we get from the CLT)

As a practioner, you must carefull consider if these assumptions are met for your data.

In [ ]:
year_2025 = pd.read_csv('./Data/2025_HR.csv')
year_2024 = pd.read_csv('./Data/2024_HR.csv')



HR_per_game_2025 = year_2025.apply(lambda row: row.HR / row.G, axis=1).values
HR_per_game_2024 = year_2024.apply(lambda row: row.HR / row.G, axis=1).values


In [ ]:
t_test_years = stats.ttest_ind(HR_per_game_2025, HR_per_game_2024, equal_var=False)
print('From 2-sample t-test:')
print('\ttest statistic t* = %0.4f' %t_test_years.statistic)
print('\tp = %0.4f' %t_test_years.pvalue)
print('\tCI = [%0.4f, %0.4f]' %(t_test_years.confidence_interval()[0], t_test_years.confidence_interval()[1]))


Here, we get a $p$-value of $0.4145 > 0.0$ and fail to reject the null.
There is not statistically significant evidence to suggest that the average number of homeruns per game differs between the two years.

But, you should still have some questions.
Even with the $p$-value, I feel like there is still a lot I don't know about the data.
What were the averages for the two years?
What does each year's distribution look like?
Are the distributions different?

This is where our earlier descriptive statistics and basic visualizations can still come in handy when analyzing data.
The inference allows for a (somewhat) regimented way to say, "yes, the data back this up" or "the data do not support this", but inference is not without its faults and just a piece of a comprehensive data analysis.